In [0]:
from pyspark.sql.functions import col, when, sum as _sum
from delta.tables import DeltaTable

raw_transaction_table = "demo.UPI_raw_transactions"
aggregated_transaction_table = "demo.UPI_aggregated_transactions"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {aggregated_transaction_table} (
        Merchant_Id STRING,
        Successfull_TRXN DECIMAL(10,2),
        Refunded_TRXN DECIMAL(10,2),
        Net_Sales DECIMAL(10,2)
    ) USING DELTA
""")

def aggregate_transactions(batch_df, batch_id):
    print('Processing Batch ', batch_id)

    aggregated_data = (
        batch_df
        .filter(col('_change_type').isin(['insert', 'update_postimage']))
        .groupBy(col('Merchant_Id'))
        .agg(
            _sum(when(col('Transaction_Status') == 'Success', col('Transaction_Amount')).otherwise(0)).alias('Successfull_TRXN'),
            _sum(when(col('Transaction_Status') == 'Refunded', col('Transaction_Amount')).otherwise(0)).alias('Refunded_TRXN')
        )
        .withColumn('Net_Sales', col('Successfull_TRXN') - col('Refunded_TRXN'))
    )

    delta_table = DeltaTable.forName(spark, aggregated_transaction_table)
    (
        delta_table.alias('target')
        .merge(
            aggregated_data.alias('source'),
            "source.Merchant_Id = target.Merchant_Id"
        )
        .whenMatchedUpdate(
            set={
                "Successfull_TRXN": "source.Successfull_TRXN + target.Successfull_TRXN",
                "Refunded_TRXN": "source.Refunded_TRXN + target.Refunded_TRXN",
                "Net_Sales": "source.Net_Sales + target.Net_Sales"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f'Batch {batch_id} processed successfully')

cdc_stream = (
    spark.readStream
    .format('delta')
    .option('readChangeData', "true")
    .table(raw_transaction_table)
)
print('Reading Stream...')
cdc_stream.writeStream.foreachBatch(aggregate_transactions).outputMode("update").start().awaitTermination()
print('Writing Stream...')

In [0]:
spark.sql(f'Select * from {aggregated_transaction_table}').display()